# P2 · Repeat the 25% sampling mixture
Run only the two missing 25% mixture seeds. Reuse verified 0%/50% runs and the original 25% seed for a three-policy comparison. Enable Internet and GPU.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json

project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)],
        check=True,
    )
os.chdir(project)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy

print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve

raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not (cache / "metadata.json").exists():
    subprocess.run([sys.executable, "-m", "acr.data.billboard.prepare"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
assert torch.cuda.is_available(), "Enable a Kaggle GPU."
repeat_root = project / "runs/p2-mixture25-repeat"
plan = json.loads((project / "configs/p2/p2-mixture25-seeds.json").read_text())
assert plan["seeds"] == [20261120, 20261121]
template = json.loads((project / plan["config"]).read_text())
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
record = {"plan": plan, "revision": revision}
repeat_root.mkdir(parents=True, exist_ok=True)
plan_file = repeat_root / "plan.json"
if plan_file.exists():
    assert json.loads(plan_file.read_text()) == record, "Existing root uses another plan/revision."
plan_file.write_text(json.dumps(record, indent=2))
for seed in plan["seeds"]:
    config = dict(template, seed=seed)
    config_file = repeat_root / f"mixture25-{seed}.json"
    config_file.write_text(json.dumps(config, indent=2))
    target = repeat_root / f"mixture25-{seed}"
    if target.exists():
        assert (target / "metrics.json").is_file() and (
            target / "complementarity.csv"
        ).is_file(), f"Incomplete run: {target}. Preserve it and use a fresh root or remove only that failed run."
        previous = json.loads((target / "run.json").read_text())
        assert previous["config"] == config and previous["git_revision"] == revision
        print("Reusing completed run", target)
    else:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "acr.training.event",
                "--config",
                str(config_file),
                "--output",
                str(target),
                "--device",
                "cuda",
            ],
            check=True,
        )
    print((target / "metrics.json").read_text())


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "acr.experiments.billboard.mixtures",
        "--runs",
        str(repeat_root),
        "--output",
        str(repeat_root / "analysis"),
    ],
    check=True,
)
print((repeat_root / "analysis/per_seed.csv").read_text())


In [ ]:
import shutil

archive = shutil.make_archive(
    "/kaggle/working/p2-mixture25-repeat-results", "zip", project, "runs/p2-mixture25-repeat"
)
print(archive)
